# 02 — Train and evaluate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SepidehNorouziMotlagh/speaker-disentanglement-grl/blob/main/notebooks/02_train_and_evaluate.ipynb)

Trains the three conditions (baseline / multitask / adversarial), runs the
independent speaker-verification attacker, sweeps attacker and
representation seeds, and produces all reported metrics, the privacy–utility
figure, and the experiment manifest under `results/`.

**Prerequisite:** run `01_extract_wavlm_embeddings.ipynb` first so that
`embeddings/{train,validation,test}.npz` exist.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/speaker-disentanglement-grl')
EMBEDDING_DIR = PROJECT_DIR / 'embeddings'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR = PROJECT_DIR / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data.dataset import load_split, load_metadata
from src.models import EmotionClassifier, SpeakerDisentanglementModel
from src.training.train import train_model, evaluate_emotion, extract_representations
from src.evaluation.pairs import generate_matched_pairs
from src.evaluation.attack import train_verification_attack

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Load data

Note the **two training loaders**: `train_loader` shuffles for optimization;
`train_eval_loader` does not, so extracted representations align with the
metadata row order used by the pair generator. This is a subtle but critical
fix — using a shuffling loader here silently misaligns pairs.

In [ ]:
train_dataset = load_split(EMBEDDING_DIR, 'train')
val_dataset = load_split(EMBEDDING_DIR, 'validation')
test_dataset = load_split(EMBEDDING_DIR, 'test')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# CRITICAL: shuffle=False so representations line up with metadata rows.
train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

metadata = {s: load_metadata(EMBEDDING_DIR, s) for s in ['train', 'validation', 'test']}

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 3. Train the three conditions (seed 42)

Model selection uses validation Macro-F1. Checkpoints are saved to
`checkpoints/{baseline,multitask,adversarial}_best.pt`.

In [ ]:
set_seed(42)
baseline_model = EmotionClassifier(input_dim=768)
baseline_model, _ = train_model(
    baseline_model, train_loader, val_loader,
    experiment_name='baseline', output_dir=CHECKPOINT_DIR,
    baseline=True, device=device,
)

In [ ]:
set_seed(42)
multitask_model = SpeakerDisentanglementModel(adversarial=False, grl_lambda=1.0)
multitask_model, _ = train_model(
    multitask_model, train_loader, val_loader,
    experiment_name='multitask', output_dir=CHECKPOINT_DIR, device=device,
)

In [ ]:
set_seed(42)
adversarial_model = SpeakerDisentanglementModel(adversarial=True, grl_lambda=1.0)
adversarial_model, _ = train_model(
    adversarial_model, train_loader, val_loader,
    experiment_name='adversarial', output_dir=CHECKPOINT_DIR, device=device,
)

## 4. Seed-42 evaluation

In [ ]:
seed42_models = {
    'baseline': (baseline_model, True),
    'multitask': (multitask_model, False),
    'adversarial': (adversarial_model, False),
}

for name, (model, is_baseline) in seed42_models.items():
    m = evaluate_emotion(model, test_loader, baseline=is_baseline, device=device)
    print(f'{name:12s} | Macro-F1 {m["macro_f1"]:.4f} | Acc {m["accuracy"]:.4f}')

## 5. Main verification attack (seed 42)

Trains an independent MLP attacker per representation on matched pairs.
Attacker init seed pinned at 42.

In [ ]:
def reps_and_pairs(model, is_baseline, pair_seed: int = 42):
    reps = {
        split: extract_representations(model, loader, baseline=is_baseline, device=device)
        for split, loader in [
            ('train', train_eval_loader),
            ('validation', val_loader),
            ('test', test_loader),
        ]
    }
    pairs = {
        split: generate_matched_pairs(
            embeddings=reps[split],
            speakers=metadata[split]['speakers'],
            sentences=metadata[split]['sentences'],
            emotions=metadata[split]['emotions'],
            n_pairs=20000 if split == 'train' else 5000,
            seed=pair_seed,
        )
        for split in ['train', 'validation', 'test']
    }
    return reps, pairs

seed42_pairs = {}
for name, (model, is_baseline) in seed42_models.items():
    _, seed42_pairs[name] = reps_and_pairs(model, is_baseline)
    X, y = seed42_pairs[name]['test']
    print(f'{name:12s} test pairs {X.shape} pos={(y==1).sum()} neg={(y==0).sum()}')

In [ ]:
for name in ['baseline', 'multitask', 'adversarial']:
    r = train_verification_attack(
        train_data=seed42_pairs[name]['train'],
        val_data=seed42_pairs[name]['validation'],
        test_data=seed42_pairs[name]['test'],
        name=name, seed=42, device=device,
    )

## 6. Attacker-init robustness (seed-42 representations)

Varies the attacker initialization seed (7, 21, 100) while keeping the
representations and pair data fixed. Confirms the observed privacy pattern
is not an artifact of one attacker init.

In [ ]:
rows = []
for attacker_seed in [7, 21, 100]:
    for name in ['baseline', 'multitask', 'adversarial']:
        r = train_verification_attack(
            train_data=seed42_pairs[name]['train'],
            val_data=seed42_pairs[name]['validation'],
            test_data=seed42_pairs[name]['test'],
            name=f'{name}_attacker_seed{attacker_seed}',
            seed=attacker_seed, device=device, verbose=False,
        )
        rows.append({'attacker_seed': attacker_seed, 'model': name,
                     'test_auc': r['test_auc'], 'test_eer': r['test_eer']})

pd.DataFrame(rows).groupby('model')[['test_auc', 'test_eer']].agg(['mean', 'std'])

## 7. Representation-seed robustness (final reported results)

Retrains all three conditions with seeds 7, 21, 100. For each, extracts
representations, builds pairs, and trains a fresh attacker **with init
seed fixed at 42** so that variation is attributed to the representation,
not the attacker.

The baseline attacker AUC/EER will have zero std across seeds — this is
expected: the baseline representation is the same frozen WavLM embedding
regardless of the emotion head seed.

In [ ]:
seed_emotion_rows = []
seed_attack_rows = []

for seed in [7, 21, 100]:
    set_seed(seed)
    bl = EmotionClassifier(input_dim=768)
    bl, _ = train_model(bl, train_loader, val_loader,
                        experiment_name=f'baseline_seed{seed}',
                        output_dir=CHECKPOINT_DIR, baseline=True,
                        device=device, verbose=False)

    set_seed(seed)
    mt = SpeakerDisentanglementModel(adversarial=False, grl_lambda=1.0)
    mt, _ = train_model(mt, train_loader, val_loader,
                        experiment_name=f'multitask_seed{seed}',
                        output_dir=CHECKPOINT_DIR, device=device, verbose=False)

    set_seed(seed)
    adv = SpeakerDisentanglementModel(adversarial=True, grl_lambda=1.0)
    adv, _ = train_model(adv, train_loader, val_loader,
                         experiment_name=f'adversarial_seed{seed}',
                         output_dir=CHECKPOINT_DIR, device=device, verbose=False)

    for name, (model, is_baseline) in {
        'baseline': (bl, True), 'multitask': (mt, False), 'adversarial': (adv, False)
    }.items():
        m = evaluate_emotion(model, test_loader, baseline=is_baseline, device=device)
        seed_emotion_rows.append({'Representation_Seed': seed, 'Model': name,
                                  'Emotion_Macro_F1': m['macro_f1'],
                                  'Emotion_Accuracy': m['accuracy']})

        _, pairs = reps_and_pairs(model, is_baseline)
        r = train_verification_attack(
            train_data=pairs['train'], val_data=pairs['validation'], test_data=pairs['test'],
            name=f'{name}_rep_seed{seed}', seed=42, device=device, verbose=False,
        )
        seed_attack_rows.append({'Representation_Seed': seed, 'Model': name,
                                 'Speaker_Verification_AUC': r['test_auc'],
                                 'Speaker_Verification_EER': r['test_eer']})

    print(f'seed {seed} done')

seed_emotion_df = pd.DataFrame(seed_emotion_rows)
seed_attack_df = pd.DataFrame(seed_attack_rows)

## 8. Merge, save, and plot

In [ ]:
final_results = seed_emotion_df.merge(
    seed_attack_df, on=['Representation_Seed', 'Model'], how='inner'
).sort_values(['Model', 'Representation_Seed']).reset_index(drop=True)

final_results.to_csv(RESULTS_DIR / 'metrics' / 'final_results_by_seed.csv', index=False)

summary_rows = []
for name in ['baseline', 'multitask', 'adversarial']:
    subset = final_results[final_results['Model'] == name]
    row = {'Model': name}
    for col in ['Emotion_Macro_F1', 'Emotion_Accuracy',
                'Speaker_Verification_AUC', 'Speaker_Verification_EER']:
        row[f'{col}_Mean'] = subset[col].mean()
        row[f'{col}_Std'] = subset[col].std(ddof=1)
    summary_rows.append(row)

final_summary = pd.DataFrame(summary_rows)
final_summary.to_csv(RESULTS_DIR / 'metrics' / 'final_results_summary.csv', index=False)
final_summary.round(4)

In [ ]:
manifest = {
    'project': 'speaker-disentanglement-grl',
    'dataset': 'CREMA-D',
    'feature_extractor': 'microsoft/wavlm-base-plus',
    'feature_extractor_frozen': True,
    'embedding_dimension': 768,
    'projection_dimension': 256,
    'emotion_classes': 6,
    'speaker_disjoint_split': {'train_speakers': 64, 'validation_speakers': 13, 'test_speakers': 14},
    'representation_training_seeds': [7, 21, 100],
    'models': ['baseline', 'multitask', 'adversarial'],
    'training': {
        'optimizer': 'AdamW', 'learning_rate': 0.001, 'weight_decay': 0.0001,
        'batch_size': 32, 'max_epochs': 30, 'early_stopping_patience': 5, 'grl_lambda': 1.0,
    },
    'speaker_verification': {
        'attacker': 'MLP', 'attacker_initialization_seed': 42,
        'pair_features': 'absolute_difference',
        'pair_matching': ['emotion', 'sentence'],
        'metrics': ['ROC-AUC', 'EER'],
    },
}
with open(RESULTS_DIR / 'metrics' / 'experiment_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)
print('Manifest saved.')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
markers = {'baseline': 'o', 'multitask': 's', 'adversarial': '^'}
for name in ['baseline', 'multitask', 'adversarial']:
    s = final_results[final_results['Model'] == name]
    row = final_summary[final_summary['Model'] == name].iloc[0]
    ax.scatter(s['Speaker_Verification_EER'], s['Emotion_Macro_F1'],
               marker=markers[name], s=55, alpha=0.55)
    ax.errorbar(
        row['Speaker_Verification_EER_Mean'], row['Emotion_Macro_F1_Mean'],
        xerr=row['Speaker_Verification_EER_Std'], yerr=row['Emotion_Macro_F1_Std'],
        fmt=markers[name], markersize=11, capsize=5, label=name.capitalize(),
    )
    ax.annotate(name.capitalize(),
                (row['Speaker_Verification_EER_Mean'], row['Emotion_Macro_F1_Mean']),
                xytext=(8, 8), textcoords='offset points')

ax.set_xlabel('Speaker Verification EER (higher = better privacy)')
ax.set_ylabel('Emotion Recognition Macro-F1 (higher = better utility)')
ax.set_title('Privacy–Utility Comparison')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'figures' / 'privacy_utility_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Output check

Verify the required artifacts exist before committing. Copy
`results/metrics/*.csv`, `results/metrics/experiment_manifest.json`, and
`results/figures/privacy_utility_comparison.png` back into your local
clone.

In [ ]:
required = [
    RESULTS_DIR / 'metrics' / 'final_results_by_seed.csv',
    RESULTS_DIR / 'metrics' / 'final_results_summary.csv',
    RESULTS_DIR / 'metrics' / 'experiment_manifest.json',
    RESULTS_DIR / 'figures' / 'privacy_utility_comparison.png',
]
for p in required:
    print(('FOUND  ' if p.exists() else 'MISSING') + '  ' + p.name)